# `compute_steady_state_spectra` — real pyglotaran result demo

This notebook builds on **Test 1** from `test_steady_state.py` and extends it to
a full pyglotaran workflow:

1. **Simulate** a time-resolved spectrum with two *independent* exponential decays
   (parallel model, 100 time points, Gaussian IRF, Gaussian spectral shapes).
2. **Fit** the data with a 2-compartment parallel decay global analysis (rates and IRF
   free; SAS estimated as CLPs).
3. **Extract** the `a_matrix` and `species_associated_spectra` from the result and call
   `compute_steady_state_spectra`.
4. **Verify** the result against the analytic expectation.

For a parallel decay the K-matrix is diagonal, but pyglotaran normalises initial
concentrations so they sum to 1 (0.5 each for 2 species), so $A_{ii} \neq 1$ in general.
The correct reference value is therefore $c_{SS,i} = A_{ii}\,\tau_i$, read directly
from the fitted A-matrix.


## 1  Imports

In [ ]:
from __future__ import annotations

import importlib

import matplotlib.pyplot as plt
import numpy as np
import pygta_local_extras.analysis.steady_state as _ss_mod
import xarray as xr
from glotaran.io import load_model, load_parameters
from glotaran.optimization.optimize import optimize
from glotaran.project.generators.generator import generate_model_yml
from glotaran.project.scheme import Scheme
from glotaran.simulation import simulate

importlib.reload(_ss_mod)
from pygta_local_extras.analysis.steady_state import compute_steady_state_spectra

print("All imports OK")

## 2  Define true (simulation) parameters

| species | rate $k_i$ | lifetime $\tau_i$ | expected $c_{SS,i}$ |
|---------|-----------|-------------------|----------------------|
| s1      | 0.5       | 2.0               | 2.0                  |
| s2      | 0.2       | 5.0               | 5.0                  |

In [ ]:
# True rates used for simulation
RATE_1 = 0.5  # τ = 2.0
RATE_2 = 0.2  # τ = 5.0
IRF_CENTER = 0.0
IRF_WIDTH = 0.1

# Analytic expected steady-state concentrations for a parallel model (A = I)
C_SS_EXPECTED = {"species_1": 1.0 / RATE_1, "species_2": 1.0 / RATE_2}
print("Expected c_SS:", C_SS_EXPECTED)

## 3  Simulate time-resolved data

In [ ]:
# ── Coordinate axes ──────────────────────────────────────────────────────────
TIME_AXIS = np.linspace(-0.5, 20.0, 100)  # 100 time points
SPECTRAL_AXIS = np.arange(600, 700, 2.0)  # 50 spectral points

SIMULATION_COORDS = {"time": TIME_AXIS, "spectral": SPECTRAL_AXIS}

# ── Simulation model (parallel + spectral shapes for SAS) ────────────────────
SIM_MODEL_YML = generate_model_yml(
    generator_name="spectral_decay_parallel",
    generator_arguments={"nr_compartments": 2, "irf": True},
)
SIM_MODEL = load_model(SIM_MODEL_YML, format_name="yml_str")

# ── Simulation parameters ─────────────────────────────────────────────────────
SIM_PARAMS_YML = f"""
rates:
  - [species_1, {RATE_1}]
  - [species_2, {RATE_2}]

irf:
  - [center, {IRF_CENTER}]
  - [width,  {IRF_WIDTH}]

shapes:
  species_1:
    - [amplitude, 1.0]
    - [location,  620]
    - [width,      20]
  species_2:
    - [amplitude, 0.8]
    - [location,  650]
    - [width,      30]
"""
SIM_PARAMS = load_parameters(SIM_PARAMS_YML, format_name="yml_str")

# ── Simulate ──────────────────────────────────────────────────────────────────
dataset = simulate(
    SIM_MODEL,
    "dataset_1",
    SIM_PARAMS,
    SIMULATION_COORDS,
    noise=True,
    noise_std_dev=5e-3,
    noise_seed=42,
)

print("Simulated dataset:", dataset)
print(f"\n  time axis:     {len(TIME_AXIS)} points  [{TIME_AXIS[0]:.2f} … {TIME_AXIS[-1]:.2f}]")
print(
    f"  spectral axis: {len(SPECTRAL_AXIS)} points [{SPECTRAL_AXIS[0]:.0f} … {SPECTRAL_AXIS[-1]:.0f} nm]"
)

In [ ]:
# ── Visualise the simulated data ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: 2D heatmap (time × wavelength)
ax = axes[0]
data_2d = dataset["data"].values  # (time, spectral) or (spectral, time) — check dims
dims = dataset["data"].dims
if dims[0] == "spectral":
    data_2d = data_2d.T  # make (time, spectral)
im = ax.pcolormesh(SPECTRAL_AXIS, TIME_AXIS, data_2d, shading="auto", cmap="RdBu_r")
plt.colorbar(im, ax=ax, label="Intensity")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Time")
ax.set_title("Simulated data (2D map)")

# Right: decay traces at selected wavelengths
ax2 = axes[1]
for wl in [620, 640, 660]:
    idx = np.argmin(np.abs(SPECTRAL_AXIS - wl))
    trace = dataset["data"].isel({dim: idx for dim in dims if dim == "spectral"}).values
    ax2.plot(TIME_AXIS, trace, label=f"{SPECTRAL_AXIS[idx]:.0f} nm")
ax2.set_xlabel("Time")
ax2.set_ylabel("Intensity")
ax2.set_title("Decay traces at selected wavelengths")
ax2.legend()

plt.suptitle("Simulated time-resolved spectrum (2 parallel decays)", fontweight="bold")
plt.tight_layout()
plt.show()

## 4  Global analysis — fit with a 2-compartment parallel model

In [ ]:
# ── Analysis model (no spectral shapes — SAS are estimated as free CLPs) ──────
FIT_MODEL_YML = generate_model_yml(
    generator_name="decay_parallel",
    generator_arguments={"nr_compartments": 2, "irf": True},
)
FIT_MODEL = load_model(FIT_MODEL_YML, format_name="yml_str")

# ── Starting parameters (slightly offset from truth to exercise the optimizer) ─
FIT_PARAMS_YML = f"""
rates:
  - [species_1, {RATE_1 * 1.2}, {{min: 0.01, max: 10.0}}]
  - [species_2, {RATE_2 * 0.8}, {{min: 0.01, max: 10.0}}]

irf:
  - [center, 0.05, {{min: -1.0, max: 1.0}}]
  - [width,  0.15, {{min: 0.01, max: 1.0}}]
"""
FIT_PARAMS = load_parameters(FIT_PARAMS_YML, format_name="yml_str")

scheme = Scheme(
    model=FIT_MODEL,
    parameters=FIT_PARAMS,
    data={"dataset_1": dataset},
    maximum_number_function_evaluations=200,
)
print(scheme.validate())

In [ ]:
result = optimize(scheme, raise_exception=True)
print(result.optimized_parameters)

## 5  Inspect the A-matrix and optimised rates

In [ ]:
ds_result = result.data["dataset_1"]

# Find the a_matrix variable (there is one megacomplex → one a_matrix)
a_matrix_vars = [v for v in ds_result.data_vars if v.startswith("a_matrix_")]
print("A-matrix variables:", a_matrix_vars)

mc_label = a_matrix_vars[0][len("a_matrix_") :]
a_mat = ds_result[a_matrix_vars[0]]
lifetimes = a_mat.coords[f"lifetime_{mc_label}"].values
species = a_mat.coords[f"species_{mc_label}"].values
rates_opt = a_mat.coords[f"rate_{mc_label}"].values

print(f"\nOptimised rates:    {dict(zip(species, rates_opt.round(4)))}")
print(f"True rates:         {{'species_1': {RATE_1}, 'species_2': {RATE_2}}}")
print(f"\nLifetimes (τ=1/k):  {dict(zip(species, lifetimes.round(4)))}")
print(f"\nA-matrix values:")
print(a_mat.values)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
a_vals = a_mat.values
im = ax.imshow(a_vals, cmap="Blues", vmin=0)
for i in range(a_vals.shape[0]):
    for j in range(a_vals.shape[1]):
        ax.text(j, i, f"{a_vals[i, j]:.4f}", ha="center", va="center", fontsize=9)
ax.set_xticks(range(len(species)))
ax.set_xticklabels(list(species))
ax.set_yticks(range(a_vals.shape[0]))
ax.set_yticklabels([f"comp {k + 1} (τ={lifetimes[k]:.2f})" for k in range(a_vals.shape[0])])
ax.set_title("A-matrix from optimised result\n(parallel model → identity)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 6  Species-Associated Spectra (SAS) from the result

In [ ]:
sas = ds_result["species_associated_spectra"]  # (spectral, species)

fig, ax = plt.subplots(figsize=(8, 4))
for sp in sas.coords["species"].values:
    sas.sel(species=sp).plot.line(x="spectral", ax=ax, label=sp)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Amplitude (a.u.)")
ax.set_title("Species-Associated Spectra (SAS) — estimated by global analysis")
ax.legend()
plt.tight_layout()
plt.show()

## 7  Compute steady-state spectra and verify against analytic expectation

For a **parallel** decay model the K-matrix is diagonal, so its eigenvectors form an
identity matrix, but pyglotaran normalises initial concentrations to sum to 1.
With 2 species each diagonal element of A is 0.5, so:

$$c_{SS,i} = A_{ii}\,\tau_i \quad \text{(off-diagonals are 0 for a parallel model)}$$

The assertion below computes the expected $c_{SS}$ directly from `A.T @ tau`,
which is exactly what `compute_steady_state_spectra` does internally.


In [ ]:
ss = compute_steady_state_spectra(result)
ds_ss = ss["dataset_1"]

print("Output variables:", list(ds_ss.data_vars))
print("Dimensions:\n ", ds_ss)

# Expected c_SS from the A-matrix: c_SS = A.T @ tau  (same formula used internally)
a_values = a_mat.values  # (n_components, n_species)
c_ss_expected = a_values.T @ lifetimes  # (n_species,)
c_ss_expected_dict = dict(zip(species, c_ss_expected))

print(
    f"\nA-matrix diagonal: {np.diag(a_values).round(4)}  (normalised so they sum to 1, not 1 each)"
)
print(f"Expected c_SS = A_diag * tau: { {k: round(v, 4) for k, v in c_ss_expected_dict.items()} }")
print()
print("Steady-state concentrations (inferred from c_SS = e_steady / SAS at peak):")
for sp in ds_ss["steady_state_spectra"].coords["species"].values:
    sas_sp = sas.sel(species=sp).values
    ess_sp = ds_ss["steady_state_spectra"].sel(species=sp).values
    peak_idx = np.argmax(np.abs(sas_sp))
    c_ss_inferred = ess_sp[peak_idx] / sas_sp[peak_idx] if sas_sp[peak_idx] != 0 else np.nan
    tau_opt_val = 1.0 / rates_opt[list(species).index(sp)]
    c_ss_ref = c_ss_expected_dict[sp]
    print(
        f"  {sp}: c_SS inferred = {c_ss_inferred:.4f}  |  A_ii*tau = {c_ss_ref:.4f}  "
        f"|  tau = {tau_opt_val:.4f}  |  match = {np.isclose(c_ss_inferred, c_ss_ref, rtol=1e-3)}"
    )

In [ ]:
# Hard assertion: c_SS_inferred == A.T @ tau  (what compute_steady_state_spectra computes)
# Does NOT assume A is the identity — uses the actual fitted A-matrix.
a_values_assert = a_mat.values  # (n_components, n_species)
c_ss_ref_assert = a_values_assert.T @ lifetimes  # (n_species,)
c_ss_ref_dict = dict(zip(species, c_ss_ref_assert))

for sp in ds_ss["steady_state_spectra"].coords["species"].values:
    sas_sp = sas.sel(species=sp).values
    ess_sp = ds_ss["steady_state_spectra"].sel(species=sp).values
    peak_idx = np.argmax(np.abs(sas_sp))
    c_ss_inferred = ess_sp[peak_idx] / sas_sp[peak_idx]
    c_ss_ref = c_ss_ref_dict[sp]
    np.testing.assert_allclose(
        c_ss_inferred,
        c_ss_ref,
        rtol=1e-3,
        err_msg=f"c_SS[{sp}] = {c_ss_inferred:.4f} != A.T@tau = {c_ss_ref:.4f}",
    )

# total == sum of per-species
total_check = ds_ss["steady_state_spectra"].sum(dim="species")
np.testing.assert_allclose(
    total_check.values,
    ds_ss["steady_state_spectrum"].values,
    rtol=1e-10,
    err_msg="total spectrum != sum of per-species spectra",
)

print("All assertions passed.")

## 8  Plot steady-state spectra

In [ ]:
spectral_axis = ds_ss["steady_state_spectrum"].coords["spectral"].values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: per-species contributions
ax = axes[0]
for sp in ds_ss["steady_state_spectra"].coords["species"].values:
    ds_ss["steady_state_spectra"].sel(species=sp).plot.line(
        x="spectral",
        ax=ax,
        label=sp,
        linewidth=1.5,
    )
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("$c_{SS,i} \\cdot \\mathrm{SAS}_i(\\lambda)$")
ax.set_title("Per-species steady-state contributions")
ax.legend()

# Right: total with per-species stacked fill
ax2 = axes[1]
sp_list = list(ds_ss["steady_state_spectra"].coords["species"].values)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
bottom = np.zeros(len(spectral_axis))
for i, sp in enumerate(sp_list):
    contrib = ds_ss["steady_state_spectra"].sel(species=sp).values
    ax2.fill_between(spectral_axis, bottom, bottom + contrib, alpha=0.5, color=colors[i], label=sp)
    bottom += contrib
ax2.plot(
    spectral_axis,
    ds_ss["steady_state_spectrum"].values,
    color="black",
    linewidth=2,
    linestyle="--",
    label="total",
)
ax2.set_xlabel("Wavelength (nm)")
ax2.set_ylabel("$e_{steady}(\\lambda)$")
ax2.set_title("Total steady-state spectrum\n(stacked species contributions)")
ax2.legend()

plt.suptitle(
    f"Steady-state spectra — parallel 2-species model\n"
    f"($k_1={RATE_1}$, $\\tau_1={1 / RATE_1:.1f}$; "
    f"$k_2={RATE_2}$, $\\tau_2={1 / RATE_2:.1f}$)",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## 9  Compare fitted vs true rates and c_SS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sp_labels = list(species)
x = np.arange(len(sp_labels))

# Left: rates
ax = axes[0]
true_rates = [RATE_1, RATE_2]
ax.bar(x - 0.2, true_rates, width=0.35, label="true", color="steelblue")
ax.bar(x + 0.2, rates_opt, width=0.35, label="fitted", color="tomato", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(sp_labels)
ax.set_ylabel("Rate $k_i$")
ax.set_title("Fitted vs true rates")
ax.legend()

# Right: c_SS = A_ii * tau
# For a 2-species parallel model pyglotaran normalises initial concentrations to sum to 1,
# so A_ii_true = 0.5 (not 1).  true c_SS = 0.5 * (1/k_i).
ax2 = axes[1]
n_species = len(sp_labels)
true_css = [1.0 / (n_species * r) for r in true_rates]  # A_ii_true = 1/n

# Infer c_SS from the steady-state spectrum
fitted_css = []
for sp in sp_labels:
    sas_sp = sas.sel(species=sp).values
    ess_sp = ds_ss["steady_state_spectra"].sel(species=sp).values
    peak_idx = np.argmax(np.abs(sas_sp))
    fitted_css.append(ess_sp[peak_idx] / sas_sp[peak_idx])

ax2.bar(
    x - 0.2, true_css, width=0.35, label=r"true $A_{ii}\,\tau_i = \tau_i/n$", color="steelblue"
)
ax2.bar(x + 0.2, fitted_css, width=0.35, label="from result", color="tomato", alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(sp_labels)
ax2.set_ylabel(r"$c_{SS,i} = A_{ii}\,\tau_i$")
ax2.set_title("True vs computed steady-state concentrations")
ax2.legend()

plt.suptitle("Validation: fitted parameters and $c_{SS}$ vs ground truth", fontweight="bold")
plt.tight_layout()
plt.show()